In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- HNN++ Unidirectional Poincare MLR (c = 1) ----------

def expmap0(u, eps=1e-15):
    # Exp map at the Poincare origin for curvature -1; u uses hyperbolic tangent norm,
    # hence x = tanh(||u||/2) * u/||u|| (the factor 1/2 comes from lambda_0 = 2).
    r = u.norm(dim=-1, keepdim=True).clamp_min(eps)
    return torch.tanh(r / 2) * u / r

def mobius_add(x, y, eps=1e-15):                       # x ⊕ y, c=1
    xy, x2, y2 = (x*y).sum(-1, keepdim=True), x.square().sum(-1, keepdim=True), y.square().sum(-1, keepdim=True)
    return ((1 + 2*xy + y2)*x + (1 - x2)*y) / (1 + 2*xy + x2*y2).clamp_min(eps)

def poincare_mlr(x, a, r, eps=1e-15):
    # a: [C, D] Euclidean direction/normal parameter at origin; magnitude scales logit;  r: [C]   signed radial distance from origin to each class hyperplane
    ahat = F.normalize(a, dim=-1)
    q = expmap0(r[:, None] * ahat)                     # q_{a,r}: closest point on hyperplane
    v = mobius_add(-q[None], x[:, None])               # -q ⊕ x                 [B,C,D]
    lam = 2 / (1 - v.square().sum(-1)).clamp_min(eps)  # λ_{-q⊕x}
    return 2*a.norm(dim=-1) * torch.asinh((v*ahat).sum(-1) * lam)

In [2]:
def expmap0_lorentz(u, eps=1e-15):                     # tangent@origin -> Lorentz, K=-1
    r = u.norm(dim=-1, keepdim=True).clamp_min(eps)
    return torch.cat([torch.cosh(r), torch.sinh(r) * u/r], dim=-1)

def lorentz_mlr(x, a, r, eps=1e-15):
    # x: [B,D+1] Lorentz points ordered as [time, spatial]; assumes curvature -1
    # a: [C,D] Euclidean spatial normal; magnitude scales the logit
    # r: [C] signed radial distance from origin to each class hyperplane
    anorm = a.norm(dim=-1).clamp_min(eps)
    alpha = torch.cosh(r) * (x[:, None, 1:] * a).sum(-1) \
          - torch.sinh(r) * anorm * x[:, None, 0]
    beta = anorm                                         # sqrt(cosh²(r)||a||² - sinh²(r)||a||²)
    return beta * torch.asinh(alpha / beta)

In [7]:
def poincare_mlr2(x, a, r, c=1., eps=1e-15): # alternative but identical
    rc = x.new_tensor(c).sqrt()
    an = a.norm(dim=-1).clamp_min(eps)
    ah = a / an[:, None]
    cx2 = (rc*x).square().sum(-1, keepdim=True)
    t = rc*r
    s = (2*(rc*x) @ ah.T * torch.cosh(t) - (1+cx2)*torch.sinh(t)) / (1-cx2).clamp_min(eps)
    return 2*an/rc * torch.asinh(s)

In [8]:
def p2l(x):  # Poincare -> Lorentz
    x2 = (x*x).sum(-1, keepdim=True)
    return torch.cat([(1+x2)/(1-x2), 2*x/(1-x2)], -1)

def l2p(x):  # Lorentz -> Poincare
    return x[..., 1:] / (x[..., :1] + 1)

In [10]:
torch.manual_seed(1)
dtype = torch.float32

B, C, D = 4, 5, 10
u = torch.randn(B, D, dtype=dtype) * .5


a = nn.Parameter(torch.randn(C, D, dtype=dtype))       # direction + logit scale
r = nn.Parameter(torch.randn(C, dtype=dtype))          # signed hyperplane radius


tensor([[-1.0047, -2.8928, -2.3547, -6.1832,  6.6908],
        [-2.2288,  0.2218,  0.1492, -6.4534,  5.0857],
        [-4.6733,  4.2764, -3.4299, -4.9759, 10.2535],
        [-6.3844,  2.9468,  1.5593, -8.2078,  7.0896]], grad_fn=<MulBackward0>)


In [11]:
x = expmap0(u)                                         # Poincare samples
logits_P = poincare_mlr(x, a/2, r) # division by two in the tangent vector is for the Lorentz Poincaré correspondence
x_L = expmap0_lorentz(u)  # lorentz sample
logits_L = lorentz_mlr(x_L, a, r)

print(logits_L)
print(logits_P)
print((logits_L - logits_P).abs().max())

tensor([[-1.0047, -2.8928, -2.3547, -6.1832,  6.6908],
        [-2.2288,  0.2218,  0.1492, -6.4534,  5.0857],
        [-4.6733,  4.2764, -3.4299, -4.9759, 10.2535],
        [-6.3844,  2.9468,  1.5593, -8.2078,  7.0896]], grad_fn=<MulBackward0>)
tensor([[-1.0047, -2.8928, -2.3547, -6.1832,  6.6908],
        [-2.2288,  0.2218,  0.1492, -6.4534,  5.0857],
        [-4.6733,  4.2764, -3.4299, -4.9759, 10.2535],
        [-6.3844,  2.9468,  1.5593, -8.2078,  7.0896]], grad_fn=<MulBackward0>)
tensor(1.9073e-06, grad_fn=<MaxBackward1>)


In [12]:
l2p(x_L) - x

tensor([[ 2.9802e-08,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.8626e-09,
          1.4901e-08,  1.4901e-08,  2.9802e-08,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  3.7253e-09,  3.7253e-09,  0.0000e+00, -3.7253e-09,
         -7.4506e-09,  0.0000e+00,  0.0000e+00, -1.4901e-08,  0.0000e+00],
        [-1.4901e-08, -3.7253e-09,  1.4901e-08, -2.9802e-08,  2.9802e-08,
          0.0000e+00, -2.9802e-08,  1.8626e-09, -3.7253e-09,  1.4901e-08],
        [ 1.4901e-08,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  1.4901e-08,  0.0000e+00,  0.0000e+00]])

In [13]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class LorentzMLR(nn.Module):
    def __init__(self, num_features, num_classes, c=1., eps=1e-15):
        super().__init__()
        # num_features: the dimension of the space components
        self.c, self.eps = c, eps
        self.a = nn.Parameter(torch.zeros(num_classes))                # signed hyperplane offset
        self.z = nn.Parameter(torch.empty(num_classes, num_features))  # normal
        self.init_weights()

    def init_weights(self):
        stdv = 1. / math.sqrt(self.z.size(1))
        nn.init.uniform_(self.z, -stdv, stdv)
        nn.init.uniform_(self.a, -stdv, stdv)
        
    def forward(self, x):
        # x: [B,D] Lorentz spatial components only; time = sqrt(c + ||x||²)
        c = x.new_tensor(self.c)
        xt = torch.sqrt(c + x.square().sum(-1, keepdim=True))
        zn = self.z.norm(dim=-1).clamp_min(self.eps)

        alpha = torch.cosh(self.a/c.sqrt()) * (x @ self.z.T) \
              - torch.sinh(self.a/c.sqrt()) * zn * xt

        return c.sqrt()*zn * torch.asinh(alpha/(c.sqrt()*zn))

In [14]:
# class PoincareMLR(nn.Module): (works great with the c = 1; explicit mobius operation)
#     def __init__(self, num_features, num_classes):
#         super().__init__()
#         self.a = nn.Parameter(torch.zeros(num_classes, num_features))  # normal: direction + scale
#         self.r = nn.Parameter(torch.zeros(num_classes))                # signed hyperplane radius
#         self.init_weights()

#     def init_weights(self):
#         stdv = 1. / math.sqrt(self.a.size(1))
#         nn.init.uniform_(self.a, -stdv, stdv)
#         nn.init.uniform_(self.r, -stdv, stdv)

#     def forward(self, x, eps=1e-15):
#         ahat = F.normalize(self.a, dim=-1) # making it equivalent for the lorentz setting. 
#         q = expmap0(self.r[:, None] * ahat, eps)
#         v = mobius_add(-q[None], x[:, None], eps)
#         lam = 2 / (1 - v.square().sum(-1)).clamp_min(eps)
#         return 2*self.a.norm(dim=-1) * torch.asinh((v*ahat).sum(-1) * lam)

class PoincareMLR(nn.Module):
    def __init__(self, num_features, num_classes, c=1., eps=1e-15):
        super().__init__()
        self.c, self.eps = c, eps
        self.a = nn.Parameter(torch.zeros(num_classes, num_features))  # normal: direction + scale (2*the lorentz tangent)
        self.r = nn.Parameter(torch.zeros(num_classes))                # signed hyperplane radius
        self.init_weights()

    def init_weights(self):
        stdv = 1. / math.sqrt(self.a.size(1))
        nn.init.uniform_(self.a, -stdv, stdv)
        nn.init.uniform_(self.r, -stdv, stdv)

    def forward(self, x):
        # x: [B,D] Poincare coordinates; ball radius = 1/sqrt(c)
        rc = x.new_tensor(self.c).sqrt()
        an = self.a.norm(dim=-1).clamp_min(self.eps)
        ah = self.a / an[:, None]

        cx2 = (rc*x).square().sum(-1, keepdim=True)
        t = rc*self.r
        s = (2*(rc*x) @ ah.T * torch.cosh(t)
             - (1+cx2)*torch.sinh(t)) / (1-cx2).clamp_min(self.eps)

        return 2*an/rc * torch.asinh(s)

In [15]:
lor_mlr = LorentzMLR(num_features= D, num_classes=C) #.double() # 1 dimention more because of the time-coordinates [match the dtype]
poin_mlr = PoincareMLR(num_features= D, num_classes=C) # .double() 

In [16]:
# copying the Lorentz parameters to the Poincaré model 
with torch.no_grad():
    poin_mlr.a.copy_(lor_mlr.z*0.5)   # Poincare normal; ×0.5 matches normal magnitude
    poin_mlr.r.copy_(lor_mlr.a)         # same signed hyperplane radius

In [18]:
lor_mlr(x_L[:, 1:]) - poin_mlr(x) # lorentz mlr expects the spatial component only

tensor([[-1.4901e-08,  5.9605e-08,  0.0000e+00,  0.0000e+00,  2.9802e-08],
        [ 6.3330e-08, -2.9802e-08,  0.0000e+00,  1.4901e-08,  0.0000e+00],
        [-1.1921e-07,  0.0000e+00,  1.1921e-07,  0.0000e+00, -8.9407e-08],
        [-5.9605e-08,  0.0000e+00, -4.4703e-08, -3.7253e-08,  0.0000e+00]],
       grad_fn=<SubBackward0>)